# This is a notebook to train agent in CatGym environment

In [ ]:
%load_ext autoreload
%autoreload 2
%env MKL_NUM_THREADS=1
%env OMP_NUM_THREADS=1
%env NUMEXPR_NUM_THREADS=1
%env MKL_DEBUG_CPU_TYPE=5

In [ ]:
import tensorflow as tf
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
seed = 30
config = ConfigProto()
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)

# Set up the environment

In [ ]:
import sys
import os
# 强制加入正确的路径
sys.path.insert(0, '/root/apps/catgym_AgPd')

import gym
from surface_seg.envs.catgym_env import MCSEnv
from surface_seg.utils.callback_new import Callback
from tensorforce.execution import Runner
import gym.wrappers
import numpy as np
import tensorforce
import copy
import json

In [ ]:
timesteps = 500
# Substitute your own directory for saving result during training
save_dir = './result_multi_env/test_catgym'
num_parallel = 32
thermal_threshold = 3

In [ ]:
def setup_env(recording=True, structure=None, structure_idx=None):
    
    # Set up gym
    MCS_gym = MCSEnv(observation_fingerprints=True, 
                     observation_forces=True,
                     permute_seed=60, 
                     save_dir = save_dir,
                     timesteps = timesteps,
                     thermal_threshold = thermal_threshold,
                     save_every_min = 1,
                     save_every = 50,
                     step_size = 0.2,                    
                    )
    
    if recording:
    # Wrap the gym to provide video rendering every 50 steps
        MCS_gym = gym.wrappers.Monitor(MCS_gym, 
                                         os.path.join(save_dir, 'vid'), 
                                         force=True,
                                        video_callable = lambda episode_id: (episode_id+1)%50==0) #every 50, starting at 51
    
    #Convert gym to tensorforce environment
    env = tensorforce.environments.OpenAIGym(MCS_gym,
                                         max_episode_timesteps=timesteps,
                                         visualize=False)
    
    return env


"""
Create a environment for checking the intial energy and thermal energy
"""
env = setup_env().environment.env
print('initial energy', env.initial_energy)
print('thermal energy', env.thermal_energy)
n =thermal_threshold
print('%dKT' %n, n * env.thermal_energy)

# Set up the agent in tensorforce

In [ ]:
from tensorforce.agents import Agent
tf.random.set_seed(seed)
agent = Agent.create(
    agent=dict(type='trpo'),
    environment=setup_env(recording=False),
    batch_size=1,  
    learning_rate=1e-3,  
    memory=50000,  
    max_episode_timesteps=timesteps,
    exploration=dict(
        type='decaying', unit='timesteps', decay='exponential',
        initial_value=0.2, decay_steps=50000, decay_rate=0.5  
    ),
    parallel_interactions=num_parallel,  
)

In [ ]:
from tensorforce.agents import Agent
tf.random.set_seed(seed)
agent = Agent.create(
    agent=dict(type='trpo'),
    environment=setup_env(recording=False),
    batch_size=1,  
    learning_rate=1e-4,  
    memory=50000,  
    max_episode_timesteps=timesteps,
    exploration=dict(
        type='decaying', unit='timesteps', decay='exponential',
        initial_value=0.8, decay_steps=50000, decay_rate=0.5  
    ),
    parallel_interactions=num_parallel,  
)
#

In [ ]:
from tensorforce.agents import Agent
tf.random.set_seed(seed)
agent = Agent.create(
    agent=dict(type='ppo'),
    environment=setup_env(recording=False),
    batch_size=32,  
    learning_rate=1e-4,  
    memory=50000,  
    max_episode_timesteps=timesteps,
    exploration=dict(
        type='decaying', unit='timesteps', decay='exponential',
        initial_value=0.8, decay_steps=50000, decay_rate=0.5  
    ),
    parallel_interactions=num_parallel,  
)

In [ ]:
# Check agent specifications
print(agent.spec)

# Run the DRL method in parallel (multiple environments)

In [ ]:
num_episode = num_parallel*300

callback = Callback(num_episode, save_dir).episode_finish

runner = Runner( 
    agent=agent,
    environments=[setup_env(recording=False) for _ in range(num_parallel)],
    num_parallel=num_parallel,
    remote='multiprocessing',
    max_episode_timesteps=timesteps,
)

"""
Multi-env training does not close after being trained for specified num_episodes.
Manual termination required.
"""
runner.run(num_episodes=num_episode, callback=callback, callback_episode_frequency=1)
runner.close()

# Save the trained agent

In [ ]:
from tensorforce.agents.agent import TensorforceJSONEncoder
from collections import OrderedDict

save_agent_dir = os.path.join(save_dir, 'saved_agent')
agent_name = 'agent'

agent.model.save(directory=save_agent_dir, filename=agent_name, format='tensorflow', append=None)
spec_path = os.path.join(save_agent_dir, agent_name + '.json')
try:
    with open(spec_path, 'w') as fp:
        spec = OrderedDict(agent.spec)
        spec['internals'] = agent.internals_spec
        spec['initial_internals'] = agent.initial_internals()
        json.dump(obj=spec, fp=fp, cls=TensorforceJSONEncoder)
except BaseException:
    try:
        with open(spec_path, 'w') as fp:
            spec = OrderedDict()
            spec['states'] = agent.spec['states']
            spec['actions'] = agent.spec['actions']
            spec['internals'] = agent.internals_spec
            spec['initial_internals'] = agent.initial_internals()
            json.dump(obj=spec, fp=fp, cls=TensorforceJSONEncoder)
    except BaseException:
        os.remove(spec_path)
        print('Agent saving failed')